In [ ]:
import pandas as pd
import numpy as np
import os
from matplotlib import pyplot as plt
%matplotlib inline 
from ccf_utils import get_area_color
import vbn_utils
import decoding_utils as du
import scipy.stats
from vbn_utils import formatFigure

In [ ]:
high_res = True
if high_res:
    plt.rcParams['figure.dpi'] = 150
    plt.rcParams['savefig.dpi'] = 300
    plt.rcParams['font.size'] = 12
    plt.rcParams['pdf.fonttype'] = 42

    plt.rcParams['figure.facecolor'] = 'white'
    plt.rcParams['axes.facecolor'] = 'white'
    plt.rcParams['savefig.facecolor'] = 'white'  # affects clipboard copy too
    plt.rcParams['savefig.transparent'] = False

In [ ]:
figure_save_dir = "/Volumes/programs/mindscope/workgroups/np-behavior/VBN Manuscript/VBN_revision_figures"

In [ ]:
#Paths to all of the useful supplemental tables and tensors
active_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor.hdf5"
stim_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_stim_table_no_filter.csv"
unit_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_units_with_responsiveness.csv"

sessions_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/vbn_s3_cache/visual-behavior-neuropixels-0.4.0/project_metadata/ecephys_sessions.csv"

In [ ]:
structure_tree = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/ccf_structure_tree_2017.csv")

In [ ]:
units = pd.read_csv(unit_table_file)
units['cortical_layer'].replace('3-Feb', '2/3', inplace=True)
units['cortical_layer'].replace('6a', '6', inplace=True)
units['cortical_layer'].replace('6b', '6', inplace=True)

## Decoding dropouts and sufficiency tests

### Generated on HPC by 'run_decoding_dropouts.py'

In [ ]:
change_decoding_dir = "/Volumes/programs/mindscope/workgroups/np-behavior/VBN_revision_decoding_dropouts/pooledChangeDecoding_basesub_active"
image_decoding_dir = "/Volumes/programs/mindscope/workgroups/np-behavior/VBN_revision_decoding_dropouts/pooledImageDecoding_basesub_active"

In [ ]:
decoding_results_base = "/Volumes/programs/mindscope/workgroups/np-behavior/VBN_revision_decoding_dropouts"
label_to_dir = {'image': 'pooledImageDecoding_basesub_active',
                'lick': 'pooledLickDecoding_basesub_active',
                'change': 'pooledChangeDecoding_basesub_active',
                'changeprechange': 'pooledChangePrechangeDecoding_basesub_active',
                'reaction_time': 'pooledReactionTimeDecoding_basesub_active',
                'visual_response': 'pooledVisualResponseDecoding_basesub_active',}

In [ ]:
subset_files = []
set_files = []
tests = []
labels = []
subset_cell_types = []
subset_layers = []
subset_regions = []
subset_clusters = []
unitsamplesizes = []
experience_levels = []

for label in ['change', 'image']:
    decoding_dir = os.path.join(decoding_results_base, label_to_dir[label])
    decoding_files = os.listdir(decoding_dir)
    for file in decoding_files:

        if 'full' in file:
            continue

        set_info = file.split('_set_')[1].split('_subset_')[0]
        subset_info = file.split('subset_')[1]
        subset_info = subset_info.replace('VISp_VISl_VISal', 'VISp-VISl-VISal').replace('VISrl_VISpm_VISam', 'VISrl-VISpm-VISam')

        subset_regions.append(subset_info.split('_')[0])
        subset_layers.append(subset_info.split('_')[1])
        subset_cell_types.append(subset_info.split('_')[2])
        subset_clusters.append(subset_info.split('_')[3])

        if ('Familiar' in subset_info) or ('Novel' in subset_info):
            exp = 'Familiar' if 'Familiar' in subset_info else 'Novel'
            experience_levels.append(exp)
            tests.append(subset_info.split('_')[5])
            unitsamplesizes.append(subset_info.split('_')[6])
        else:
            experience_levels.append('all')
            tests.append(subset_info.split('_')[4])
            unitsamplesizes.append(subset_info.split('_')[5])

        set_file = [os.path.join(decoding_dir, f) for f in decoding_files if (f'_set_{set_info}' in f) and 'full' in f and f.split('_')[-4] == unitsamplesizes[-1]]
        if len(set_file) == 0:
            set_file = [None]

        subset_file = os.path.join(decoding_dir, file)
        if len(subset_file)>=260:
            subset_file = r"\\?\UNC" + os.path.join(decoding_dir, subset_file)[1:]

        subset_files.append(subset_file)
        set_files.append(set_file[0])
        labels.append(label)

data_table = pd.DataFrame({
    'subset_file': subset_files,
    'set_file': set_files,
    'test': tests,
    'label': labels,
    'subset_region': subset_regions,
    'subset_layer': subset_layers,
    'subset_cell_type': subset_cell_types,
    'subset_cluster': subset_clusters,
    'unitsample_size': np.array(unitsamplesizes).astype(int),
    'experience': experience_levels
})


In [ ]:
from notebook_utils import DiD_test, bootstrapped_diff_ci

### Calculate dropout and sufficiency scores for each neuron subset, comparing performance to the full population with matched unitsamplesize

In [ ]:
binsize = 10
time = np.arange(binsize, 750+binsize, binsize)

subsets_to_plot = ['VISall_full_100', 'VISall_23_RS', 'VISall_4_RS', 'VISall_5_RS', 'VISall_6_RS', 'VISall_all_FS', 'VISall_all_SST', 'VISp-VISl-VISal_full_50', 
                   'VISp-VISl-VISal_all_SST', 'VISrl-VISpm-VISam_all_SST', 'VISall_full_7', 'VISall_all_VIP', ]
subset_labels = ['full pop' if 'full' in s else s for s in subsets_to_plot]
colors = ['k', 'teal', 'teal', 'teal', 'teal', 'red', 'dodgerblue', 'k', 'dodgerblue', 'dodgerblue', 'k', 'orchid',]
alphas = [1, 1, 0.75, 0.5, 0.25, 1, 1, 1, 1, 1, 1, 1]
samplesizes = [40]*10 + [7]*2 
axinds = [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 2, 2]

plot_diff = False
plot_chance = True

index100 = list(time).index(100)
for decoding_label in ['change', 'image']:
    for test in ['dropout', 'sufficiency']:
        if test == 'sufficiency':
            fig, ax = plt.subplots(1, 3, gridspec_kw={'width_ratios': [7, 4, 2]})
        else:
            fig, dax = plt.subplots()
            ax = [dax, dax, dax]
        for ind, (subset, color, alpha, samplesize) in enumerate(zip(subsets_to_plot, colors, alphas, samplesizes)):
            
            #always makes sense to use 100 sample size for dropouts
            if test == 'dropout':
                samplesize = 40
                if 'full' in subset and ind>0:
                    continue
            
            subs_at_100 = []
            sets_at_100 = []
            for iexp, experience in enumerate(['Familiar', 'Novel']):
                if 'full' in subset:
                    subset_info = data_table[
                        (data_table['subset_region']==subset.split('_')[0]) & 
                        (data_table['unitsample_size']==samplesize) &
                        (data_table['test']==test) &
                        (data_table['label']==decoding_label) &
                        (data_table['experience']==experience)
                        ]
                    subset_file = subset_info['set_file'].values
                    subset_info = subset_info.iloc[[0]]
                else:
                    subset_info = data_table[
                        (data_table['subset_region']==subset.split('_')[0]) & 
                        (data_table['subset_layer']==subset.split('_')[1]) & 
                        (data_table['subset_cell_type']==subset.split('_')[2]) & 
                        (data_table['unitsample_size']==samplesize) &
                        (data_table['test']==test) &
                        (data_table['label']==decoding_label) &
                        (data_table['experience']==experience)
                        ]
                
                    subset_file = subset_info['subset_file'].values
                    if len(subset_file)>1:
                        print(f'found more than one compatible file for subset: {subset}')
                        break

                subset_data = np.load(subset_file[0])
                set_data = np.load(subset_info['set_file'].values[0])

                subset_at_100 = subset_data[:, index100]
                set_at_100 = set_data[:, index100]

                if plot_diff:
                    mean_diff = subset_at_100.mean() - set_at_100.mean()
                    ci95 = bootstrapped_diff_ci(subset_at_100, set_at_100)
                else:
                    mean_diff = subset_at_100.mean()
                    ci95 = np.percentile(subset_at_100, [2.5, 97.5])

                error = np.std(subset_at_100)

                subs_at_100.append(subset_at_100)
                sets_at_100.append(set_at_100)

                ax[axinds[ind]].errorbar([ind + iexp*0.25], [mean_diff], yerr=error, fmt='o', color=color, alpha=alpha)
                pval = scipy.stats.ranksums(subset_at_100, set_at_100, nan_policy='omit')
                print(f'{subset} sub vs set: {pval}')

            if plot_diff:
                DiD_pval = DiD_test(subs_at_100[0], sets_at_100[0], subs_at_100[1], sets_at_100[1])
                print(f'DiD p-value for {subset}: {DiD_pval}')
        if test == 'sufficiency':
            for ia, a in enumerate(ax):
                a_inds = np.where(np.array(axinds)==ia)[0]
                if len(a_inds)>0:
                    a.set_xticks(np.arange(len(a_inds))+a_inds[0] + 0.125)
                    a.set_xticklabels(np.array(subset_labels)[a_inds], rotation=90)
                    a.set_xlim(a.get_xticks()[0]-0.25, a.get_xticks()[-1]+0.25)

                    if plot_chance:
                        chance = 0.125 if decoding_label=='image' else 0.5
                        a.axhline(chance, ls='dotted', color='k')
                    formatFigure(fig, a)
        else:
            dax.set_xticks([sind for sind, s in enumerate(subset_labels) if (sind==0 or 'full' not in s)])
            dax.set_xticklabels([s for sind, s in enumerate(subset_labels) if (sind==0 or 'full' not in s)], rotation=90)
            if plot_chance:
                chance = 0.125 if decoding_label=='image' else 0.5
                dax.axhline(chance, ls='dotted', color='k')
            formatFigure(fig, dax)

        
        delta_upper = '\u0394' if plot_diff else ''
        ylabel = f'{delta_upper} accuracy: \n{test} - full population' if plot_diff else f'{test}'
        ax[0].set_ylabel(ylabel)
        fig.suptitle(f'{test} - {decoding_label}')
        fig.tight_layout()
        

### Look at time lag between familiar and novel image decoding for each subset

In [ ]:
from notebook_utils import (
    get_sigmoidfit_midpoint
)

In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="overflow encountered in exp",
    category=RuntimeWarning
)

plt.rcParams['font.size'] = 14
binsize = 10
time = np.arange(binsize, 750+binsize, binsize)

subsets_to_plot = ['VISall_23_RS', 'VISall_4_RS', 'VISall_5_RS', 'VISall_6_RS', 'VISall_all_FS', 'VISall_all_SST', 'VISp-VISl-VISal_all_SST', 'VISrl-VISpm-VISam_all_SST', 'VISall_all_VIP',]
colors = ['teal', 'teal', 'teal', 'teal', 'red', 'dodgerblue', 'dodgerblue', 'dodgerblue', 'orchid']
alphas = [1, 0.75, 0.5, 0.25, 1, 1, 1, 1, 1]
samplesizes = [40]*8 + [7] 
axind = [0,0,0,0,0,0,1,1,2]

index100 = list(time).index(100)
decoding_label = 'change'
test = 'dropout'
for decoding_label in ['image',]:
    for test in ['dropout', 'sufficiency']:
        all_fig, all_ax = plt.subplots(1,3, gridspec_kw={'width_ratios': [6,2,1]})
        all_fig.suptitle(f'{decoding_label} {test}')
        for ind, (subset, color, alpha, samplesize) in enumerate(zip(subsets_to_plot, colors, alphas, samplesizes)):
            
            if test == 'dropout':
                samplesize = 40
            
            subs_at_100 = []
            sets_at_100 = []
            exp_latencies = []

            fig, ax = plt.subplots()
            fig.suptitle(f'{subset} {test}')
            for iexp, experience in enumerate(['Familiar', 'Novel']):
                subset_info = data_table[
                    (data_table['subset_region']==subset.split('_')[0]) & 
                    (data_table['subset_layer']==subset.split('_')[1]) & 
                    (data_table['subset_cell_type']==subset.split('_')[2]) & 
                    (data_table['unitsample_size']==samplesize) &
                    (data_table['test']==test) &
                    (data_table['label']==decoding_label) &
                    (data_table['experience']==experience)
                    ]
                
                subset_file = subset_info['subset_file'].values
                if len(subset_file)>1:
                    print(f'found more than one compatible file for subset: {subset}')
                    break

                subset_data = np.load(subset_file[0])
                set_data = np.load(subset_info['set_file'].values[0])

                subset_latencies = np.array([get_sigmoidfit_midpoint(time, sd) for sd in subset_data])
                exp_latencies.append(subset_latencies[:,0])

                mean_latency = np.nanmean(subset_latencies[:,0])
               
                error = np.nanstd(subset_latencies[:,0])

                ax.plot(time, subset_data.T, color=['b', 'r'][iexp], alpha=0.1)
                ax.plot(subset_latencies[:,0], subset_latencies[:,1], 'o', color=['b','r'][iexp], alpha=0.1)
                all_ax[axind[ind]].errorbar([ind + iexp*0.25], [mean_latency], yerr=error, fmt='o', color=color, alpha=alpha)

            diffs = exp_latencies[0] - exp_latencies[1]
            F0 = np.mean(diffs[~np.isnan(diffs)]<=0)
            lat_pval = 2 * min(F0, 1 - F0)
            
            print(f'latency pval for {subset}: {lat_pval}')
            ax.set_xlim(0, 100)
            ax.set_xlabel('Time from stimulus (ms)')
            ax.set_ylabel('Decoding accuracy')
            vbn_utils.formatFigure(fig, ax)

        for ia, a in enumerate(all_ax):
            a_inds = np.where(np.array(axind)==ia)[0]
            a.set_xticks(np.arange(len(a_inds))+a_inds[0] + 0.125)
            a.set_xticklabels(np.array(subsets_to_plot)[a_inds], rotation=90)
            a.set_xlim(a.get_xticks()[0]-0.25, a.get_xticks()[-1]+0.25)
            formatFigure(all_fig, a)

        all_ax[0].set_ylabel('Decoding latency (ms)')
        all_fig.suptitle(f'{test} - {decoding_label}')
        all_fig.tight_layout()


## GLM predictions

In [ ]:
cluster_to_color_mapping =  {
    "On-trans":  "#673092",
    "On-sust":   "#0353a7",
    "Off-trans": "#0391d9",
    "Off-sust":  "#57cfbd",
    "Stim-supp": "#05a05a",
    "Running":   "#fea603",
    "Licking":   "#fa6a50",
    "Lick ant":  "#dc153c",
    "non-coding": "#888888",
}


### Generated on HPC by 'run_GLM_prediction_psths.py'

In [ ]:
data_dir = "/Volumes/programs/mindscope/workgroups/np-behavior/VBN_revision_glm_prediction_psths"
unit_files = os.listdir(data_dir)

In [ ]:
unitids = []
num_hits = []
num_misses = []
num_nonchange_licks = []
num_nonchange_no_licks = []
predicted_psths = []
predicted_sems = []
psths = []
psth_sems = []
for unit_file in unit_files:

    unit_id = int(unit_file.split('_')[0])

    unit_data = np.load(os.path.join(data_dir, unit_file))
    unitids.append(unit_id)
    trial_counts = unit_data['trial_counts']
    num_hits.append(trial_counts[0])
    num_misses.append(trial_counts[1])
    num_nonchange_licks.append(trial_counts[2])
    num_nonchange_no_licks.append(trial_counts[3])
    predicted_psths.append(unit_data['prediction'])
    psths.append(unit_data['psth'])
    predicted_sems.append(unit_data['prediction_sem'])
    psth_sems.append(unit_data['psth_sem'])

glm_results = pd.DataFrame({'unit_id': unitids, 'num_hits': num_hits, 'num_misses': num_misses, 'num_nonchange_licks': num_nonchange_licks, 'num_nonchange_no_licks': num_nonchange_no_licks, 'predicted_psth': predicted_psths, 'psth': psths, 'psth_sem':psth_sems, 'predicted_sems':predicted_sems})


In [ ]:
glm_results.to_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/unit_glm_predicted_psths.csv")

In [ ]:
from notebook_utils import variance_explained

glm_results['variance_explained'] = glm_results.apply(variance_explained, axis=1)

In [ ]:
glm_results = glm_results.merge(units[['unit_id', 'cluster_labels_new', 'SST', 'VIP', 'RS', 'FS', 'structure_acronym']], on='unit_id')

In [ ]:
cluster_labels = ['On-trans', 'On-sust', 'Off-trans', 'Off-sust', 'Stim-supp', 'Lick ant', 'Licking', 'Running', 'action 4', 'action 5', 'action 6', 'non-coding']

# example cells with representative variance explained
exemplar_cell_ids =  [
    1126282659,
    1177946509,
    1179687443,
    1094999579,
    1100910805,
    1094991649,
    1095005619,
    1100926769,
]

boxfig, boxax = plt.subplots()
for cluster in np.arange(1,13):

    cluster_data = glm_results[(glm_results['cluster_labels_new']==cluster)]
    print(f'{cluster} {cluster_data["variance_explained"].median()}')

    color = cluster_to_color_mapping.get(cluster_labels[cluster-1], '#888888')
    boxax.boxplot(cluster_data['variance_explained'], positions=[cluster], widths=0.5, showfliers=False, patch_artist=True, whis=[5,95],
                boxprops=dict(facecolor=color, color=color),
                whiskerprops=dict(color=color),
                capprops=dict(color=color),
                medianprops=dict(color='w', linewidth=2),
            )


    cluster_data = cluster_data[cluster_data['variance_explained'].between(0.7, 0.85)].sort_values(by='variance_explained', ascending=False) #get representative examples
    if cluster_data.empty:
        continue
    
    for ir, row in cluster_data.iloc[:10].iterrows():
        if not row['unit_id'] in exemplar_cell_ids:
            continue
        fig, ax = plt.subplots()
        fig.suptitle(f'{cluster_labels[cluster-1]}')
        fig.set_size_inches(8,5)
        ax.plot(row['predicted_psth']*40, color='k')
        ax.fill_between(np.arange(len(row['predicted_psth'])), row['predicted_psth']*40 + row['predicted_sems']*40, row['predicted_psth']*40 - row['predicted_sems']*40, alpha=0.3, color='k')
        ax.plot(row['psth']*40, color='orange')
        ax.fill_between(np.arange(len(row['psth'])), row['psth']*40 + row['psth_sem']*40, row['psth']*40 - row['psth_sem']*40, alpha=0.3, color='orange')
        ax.axvspan(40, 80, alpha=0.1, color='k', lw=0)
        ax.axvspan(120, 160, alpha=0.1, color='k', lw=0)
        ax.set_xticks(np.array([0, 10, 40, 50, 80, 90, 120, 130]) + 10)
        ax.set_xticklabels([0, 250]*4)
        formatFigure(fig,ax)
        [ax.text(40*ind + 20, ax.get_ylim()[1], ['hit', 'miss', 'false alarm', 'correct reject'][ind], ha='center', va='top') for ind in range(4)]
        ax.set_xlabel('Time from stimulus (ms)')
        ax.set_ylabel('Firing Rate (Hz)')
        fig.savefig(os.path.join(figure_save_dir, f'GLM_PSTH_variance_explained_{cluster_labels[cluster-1]}_{row["unit_id"]}_test.pdf'))

boxax.set_xticks(np.arange(1,13))
boxax.set_xticklabels(cluster_labels, rotation=90)
boxax.set_ylabel('Trial-averaged variance explained')
boxax.set_xlim(0.5, 8.5)

boxfig.savefig(os.path.join(figure_save_dir, f'GLM_PSTH_variance_explained_bycluster_boxplot_test.pdf'))


## Examining feature weights on change decoders

In [ ]:
import h5py
import math
import warnings

import sklearn
from sklearn.svm import LinearSVC
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
import decoding_utils as du

In [ ]:
stim_table = pd.read_csv(stim_table_file)
unit_table = pd.read_csv(unit_table_file)
unitData = h5py.File(active_tensor_file)
sessions = pd.read_csv(sessions_table_file)

In [ ]:
def decodeSingleClass(sp, y, nCrossVal, model, unitSampleSize, unitIDs):
    '''
    sp: units x trials
    y: labels 
    nCrossVal: number of cross validation splits to run
    unitIDs: list of unit IDs for sp
    '''
    unitIDs = np.array(unitIDs)
    nUnits = sp.shape[0]
    warnings.filterwarnings('ignore')
    summary = {sampleSize: 
                {metric: [] for metric in  ('trainAccuracy',
                                            'featureWeights',
                                            'accuracy',
                                            'prediction',
                                            'confidence', 
                                            'balanced_accuracy',
                                            'unit_ids')}
                for sampleSize in unitSampleSize}
    
    for sampleSize in unitSampleSize:
        if nUnits < sampleSize:
            continue
        if sampleSize>1:
            if sampleSize==nUnits:
                nSamples = 1
                unitSamples = [np.arange(nUnits)]
            else:
                # >99% chance each neuron is chosen at least once
                nSamples = int(math.ceil(math.log(0.01)/math.log(1-sampleSize/nUnits)))
                unitSamples = [np.random.choice(nUnits,sampleSize,replace=False) for _ in range(nSamples)]
        elif sampleSize==1:
            nSamples = nUnits
            unitSamples = [[i] for i in range(nUnits)]
        else: #if sampleSize<1, just run 1 iteration with all the units available
            nSamples = 1
            unitSamples = [np.arange(nUnits)]

        for metric in summary[sampleSize]:
            summary[sampleSize][metric].append([])

        for unitSamp in unitSamples:
            cv = du.trainDecoder(model,sp[unitSamp].T,y,nCrossVal)
            summary[sampleSize]['trainAccuracy'][-1].append(np.mean(cv['train_score']))
            summary[sampleSize]['featureWeights'][-1].append(np.mean(cv['coef'],axis=0).squeeze())
            summary[sampleSize]['accuracy'][-1].append(np.mean(cv['test_score']))
            summary[sampleSize]['prediction'][-1].append(cv['predict'])
            summary[sampleSize]['confidence'][-1].append(cv['decision_function'])
            summary[sampleSize]['balanced_accuracy'][-1].append(sklearn.metrics.balanced_accuracy_score(y.astype(bool), cv['predict'].astype(bool)))
            summary[sampleSize]['unit_ids'][-1].append(unitIDs[unitSamp])

        for metric in summary[sampleSize]:
            if metric == 'prediction':
                summary[sampleSize][metric][-1] = scipy.stats.mode(summary[sampleSize][metric][-1],axis=0)[0][0]
            elif metric not in ('featureWeights', 'unit_ids'):
                summary[sampleSize][metric][-1] = np.median(summary[sampleSize][metric][-1],axis=0)
    
    warnings.filterwarnings('default')
    
    return summary


In [ ]:
regions = ('VISall', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'LGd', 'LP', 'SCMRN', 'Hipp')

### To generate flash metrics (skip to load pre-computed flash metrics)

In [ ]:
sessions = sessions[sessions['abnormal_histology'].isnull() & sessions['abnormal_activity'].isnull()]
len(sessions)

In [ ]:
unitSampleSize = [40,]
respWin = slice(20,100)

change_decoder_results = {s: {r:[] for r in regions} for s in sessions['ecephys_session_id'].values}
for count, (isess, session) in enumerate(sessions.iterrows()):
    if count%10==0:
        print(count)

    sessionId = session['ecephys_session_id']

    stim = stim_table[(stim_table['session_id']==sessionId) & stim_table['active']].reset_index()

    units = unit_table.set_index('unit_id').loc[unitData[str(sessionId)]['unitIds'][:]]
    spikes = unitData[str(sessionId)]['spikes']
    highQuality = du.apply_unit_quality_filter(units, no_abnorm=False)

    for region in regions:
        inRegion = du.getUnitsInRegion(units,region)
        final_unit_filter = highQuality & inRegion

        if np.sum(final_unit_filter)<10:
            continue

        sp = np.zeros((final_unit_filter.sum(),spikes.shape[1],spikes.shape[2]),dtype=bool)
        unitIDs = []
        for i,u in enumerate(np.where(final_unit_filter)[0]):
            sp[i]=spikes[u,:,:]
            unitIDs.append(unitData[str(sessionId)]['unitIds'][:][u])

        flash_response = sp[:, :, respWin].mean(axis=2)

        model = LinearSVC(C=1.0,max_iter=int(1e4), class_weight='balanced')
        change_results = decodeSingleClass(flash_response, stim['is_change'].values, 5, model, unitSampleSize, unitIDs)
        change_decoder_results[sessionId][region].append(change_results)

In [ ]:
dfs = []
for sessionId, decoderdata in change_decoder_results.items():
    for region, regiondata in decoderdata.items():
        if len(regiondata)>0 and len(regiondata[0][40]['featureWeights'])>0:
            dfs.append(pd.DataFrame({'unit_id':np.array(regiondata[0][40]['unit_ids'][0]).flatten(), 
                                    'feature_weights': np.array(regiondata[0][40]['featureWeights'][0]).flatten(),
                                    'region': region,
                                    'sessionId': sessionId}))
unit_decoder_df = pd.concat(dfs, ignore_index=True)


In [ ]:
unit_decoder_df = unit_decoder_df.merge(unit_table[['unit_id', 'cluster_labels_new']], on='unit_id', )

In [ ]:
cluster_to_color_mapping =  {
    "On-trans":  "#673092",
    "On-sust":   "#0353a7",
    "Off-trans": "#0391d9",
    "Off-sust":  "#57cfbd",
    "Stim-supp": "#05a05a",
    "Running":   "#fea603",
    "Licking":   "#fa6a50",
    "Lick ant":  "#dc153c",
    "non-coding": "#888888",
}


In [ ]:
plt.rcParams['figure.dpi'] = 300

In [ ]:
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'  # affects clipboard copy too
plt.rcParams['savefig.transparent'] = False

In [ ]:
cluster_labels = ['On-trans', 'On-sust', 'Off-trans', 'Off-sust', 'Stim-supp', 'Lick ant', 'Licking', 'Running', 'action 4', 'action 5', 'action 6', 'non-coding']
clusters_to_plot = list(np.arange(1,9)) + [12,]
clusters_labels_to_plot = [cluster_labels[ind-1] for ind in clusters_to_plot]

for region in regions + ['all',]:
    fig, ax = plt.subplots(1,2)
    fig.set_size_inches(12,6)
    fig.suptitle(region)

    if region == 'all':
        region_df = unit_decoder_df
    else:
        region_df = unit_decoder_df[unit_decoder_df['region'] == region]

    region_df_cluster_pivot = region_df.pivot_table(index=['sessionId', 'cluster_labels_new'], values='feature_weights')
    cluster_session_vals = []
    for c in clusters_to_plot:
        if (not c in region_df['cluster_labels_new'].unique()):
            cluster_session_vals.append([np.nan])
        else:
            cluster_session_vals.append(region_df_cluster_pivot.xs(c, level=1)['feature_weights'].values)

    cluster_session_vals = [c if len(c)>=3 else [np.nan] for c in cluster_session_vals]

    for ic, (c, clabel) in enumerate(zip(clusters_to_plot, clusters_labels_to_plot)):
        color = cluster_to_color_mapping.get(clabel, "#888888")
        ax[0].boxplot(cluster_session_vals[ic], positions=[ic], widths=0.5, showfliers=False, patch_artist=True,
                boxprops=dict(facecolor=color, color=color),
                whiskerprops=dict(color=color),
                capprops=dict(color=color),
                medianprops=dict(color='w', linewidth=2),
            )
    ax[0].set_xlabel('Cluster ID')
    ax[0].set_xticks(np.arange(len(clusters_to_plot)))
    ax[0].set_xticklabels(clusters_labels_to_plot, rotation=90)
    ax[0].set_ylabel('Feature Weight')
    ax[0].axhline(0, ls='dotted', c='k')
    vbn_utils.plot_comparison_matrix(*cluster_session_vals, ax=ax[1], labels=clusters_labels_to_plot, cmap='PiYG', colorbar=True, binarize=True)

    plt.tight_layout()